# Command line usage

SwarmPAL can be used directly from the command line by supplying data and process configuration YAML files.

In [ ]:
%%bash
swarmpal --help

In [ ]:
%%bash
swarmpal batch --help

## Flexible operation

The CLI can be used in different ways:

- A) Fetch data from VirES/HAPI and apply a process  
  - Generate output data with no interim files
- B) Fetch data from VirES/HAPI, store it, then apply a process separately  
  - Useful if you want to separate steps, e.g. gather all in the inputs first, then separately apply the processing which occurs locally (where you might iterate to experiment with options, or run the processes in parallel)
- C) Use local data and apply a process
  - Useful for working with data not available via VirES/HAPI

## A) Fetch data from VirES and apply a process

Write a configuration file like the following (note it has two parts, `data_params` and `process_params`)

:::{literalinclude} configs/FAC_fetch_and_process.yml
:language: yaml
:caption: configs/FAC_fetch_and_process.yml
:::

Provide `swarmpal batch` with the configuration file and desired output file name

In [ ]:
%%bash
swarmpal batch --overwrite configs/FAC_fetch_and_process.yml temp/FAC_processed_A.nc

The output data has been saved in the file `FAC_processed_A.nc` which we might then interrogate in an interactive session, or can generate a quicklook:

In [ ]:
%%bash
swarmpal quicklook --overwrite temp/FAC_processed_A.nc temp/FAC_quicklook_A.png

![](temp/FAC_quicklook_A.png)

## B) Fetch data from VirES, store it, then apply a process separately

Write a configuration file like the following that only specifies the `data_params`

:::{literalinclude} configs/FAC_fetch_inputs.yml
:language: yaml
:caption: configs/FAC_fetch_inputs.yml
:::

In this example, we will override the start and end times using the `--time` option:

In [ ]:
%%bash
swarmpal batch --overwrite --time "2025-02-02T00:00:00" "2025-02-03T00:00:00" configs/FAC_fetch_inputs.yml temp/FAC_inputs_B.nc

:::{note}
NB: `swarmpal fetch-data` is identical in behaviour if there are no processes in the config file - maybe we should remove it to avoid confusion about its purpose, since there is no matching "apply-process" command.
:::

The inputs have been retrieved from VirES and stored as `FAC_inputs_B.nc`. Next we can apply the process:

:::{literalinclude} configs/FAC_apply_process.yml
:language: yaml
:caption: configs/FAC_apply_process.yml
:::

In [ ]:
%%bash
swarmpal batch --overwrite configs/FAC_apply_process.yml temp/FAC_processed_B.nc
swarmpal quicklook --overwrite temp/FAC_processed_B.nc temp/FAC_quicklook_B.png

![](temp/FAC_quicklook_B.png)

## C) Use local data and apply a process

:::{literalinclude} configs/FAC_apply_process_local_file.yml
:language: yaml
:caption: configs/FAC_apply_process_local_file.yml
:::

Note that in this case, we also run an experimental process `EXP_LocalForwardMagneticModel` to provide the CHAOS model predictions, computed locally (whereas before they were computed on VirES).

(This example usage is skipped since you must supply the CDF file)

In [ ]:
# %%bash
# swarmpal batch --overwrite configs/FAC_apply_process_local_file.yml temp/FAC_processed_C.nc
# swarmpal quicklook --overwrite temp/FAC_processed_C.nc temp/FAC_quicklook_C.png
# display(Image('temp/FAC_quicklook_C.png'))

## Example: TFA

:::{literalinclude} configs/TFA.yml
:language: yaml
:caption: configs/TFA.yml
:::

In [ ]:
%%bash
swarmpal batch --overwrite configs/TFA.yml TFA_processed.nc
swarmpal quicklook --overwrite TFA_processed.nc temp/TFA_quicklook.png

:::{warning}
It looks like auxiliaries are not being included when we specify them in the config file
:::

![](temp/TFA_quicklook.png)

## Example: DSECS

:::{literalinclude} configs/DSECS.yml
:language: yaml
:caption: configs/DSECS.yml
:::

In [ ]:
# %%bash
# swarmpal batch --overwrite configs/DSECS.yml temp/DSECS_processed.nc
# swarmpal quicklook --overwrite temp/DSECS_processed.nc temp/DSECS_quicklook.png

In [ ]:
# display(Image("temp/DSECS_quicklook.png"))

To separate data fetching from process application, you could use:

```
swarmpal fetch-data configs/DSECS.yml temp/DSECS_inputs.nc
```

Then:

```
swarmpal batch DSECS_apply_process.yml temp/DSECS_output.nc
```

with a configuration file like:

```
data_params:
  - provider: file
    filename: "temp/DSECS_inputs.nc"
    filetype: "netcdf"
    dataset: "SW_OPER_MAGA_LR_1B"
  - provider: file
    filename: "temp/DSECS_inputs.nc"
    filetype: "netcdf"
    dataset: "SW_OPER_MAGC_LR_1B"

process_params:
  - process_name: DSECS_Preprocess
  - process_name: DSECS_Analysis
```

(Note that both input datasets, MAGA and MAGC, are stored in one .nc file)